# FordRetain — Classificação Preditiva (Base 2)

**Regra crítica:** Use APENAS variáveis disponíveis no momento da compra.

Proibido usar: `total_revisoes`, `km`, `intervalos`, `dias_desde_ultima_revisao`, datas de serviço.

Qualquer variável com informação do comportamento futuro = **DATA LEAKAGE** = modelo inválido.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib
import sys
import os
sys.path.append('../src')
from preprocessamento import carregar_dados, construir_base2

os.makedirs('../outputs', exist_ok=True)

base1_perfis = pd.read_csv('../models/base1_com_perfis.csv')
df = carregar_dados('../dados/vin_share_Desafio_02.xlsx')
base2 = construir_base2(df)

print(f'Base 2 shape: {base2.shape}')
base2.head()

In [ ]:
le_modelo = LabelEncoder()
base2 = base2.copy()
base2['modelo_enc'] = le_modelo.fit_transform(base2['modelo'].fillna('Desconhecido'))

df_sup = base2.reset_index(drop=True).copy()
df_sup['perfil'] = base1_perfis.reset_index(drop=True)['perfil'].values
df_sup = df_sup.dropna(subset=['perfil', 'mes_compra'])

print(f'Dataset supervisionado: {df_sup.shape}')
print(df_sup['perfil'].value_counts())

In [ ]:
features_base2 = ['modelo_enc', 'ano_modelo', 'mes_compra', 'trimestre_compra', 'dias_venda_entrega']

X = df_sup[features_base2].fillna(0)
y = df_sup['perfil']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('=== Relatório de Classificação ===')
print(classification_report(y_test, y_pred))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
plt.title('Matriz de Confusão — FordRetain Classifier')
plt.tight_layout()
plt.savefig('../outputs/08_matriz_confusao.png', dpi=150)
plt.show()

In [ ]:
importancias = pd.Series(clf.feature_importances_, index=features_base2).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
importancias.plot(kind='bar', color='#1F3A6E')
plt.title('Importância das Features — RandomForest')
plt.ylabel('Importância')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../outputs/09_feature_importance.png', dpi=150)
plt.show()
print(importancias)

In [ ]:
def calcular_score_risco(modelo_clf, X_input):
    """Score de risco 0-100. 100 = abandono certo."""
    proba = modelo_clf.predict_proba(X_input)
    classes = list(modelo_clf.classes_)
    idx = classes.index('Abandono') if 'Abandono' in classes else 0
    return (proba[:, idx] * 100).round(0).astype(int)

scores = calcular_score_risco(clf, X_test.iloc[:5])
print('Scores dos primeiros 5 clientes do teste:')
for i, (s, r, p) in enumerate(zip(scores, y_test.iloc[:5], y_pred[:5])):
    print(f'  Cliente {i+1}: Score={s:3d} | Real={r} | Previsto={p}')

joblib.dump(clf, '../models/classificador_perfil.pkl')
joblib.dump(le_modelo, '../models/encoder_modelo.pkl')
print('\nModelos salvos.')

In [ ]:
cv_scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
print(f'Cross-validation (5-fold):')
print(f'  Acurácia média: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')